# DSA format test: one reasoning step per line

The CoT checkpoint DSA starts from was trained on LOTUS's format, one step per line
(`question\n<<a>>\n<<b>>\n### N`). DSA's explicit route joins steps with spaces, and after DSA
training the explicit route drops from 25/64 to 13–17/64 on dev64 (`docs/AUDIT_20261025.md`, finding 1).

This notebook retrains the **learned** data-scale arm with newline-joined steps and changes nothing else:
the same 512 questions in the same order, seed 7, K=8 FLT recurrence, 1,024 updates. The original arms
ran on a Colab **T4**, so choose **Runtime → Change runtime type → T4 GPU**. On a T4 the run should
start from the original initial state, and the comparison with the saved space-joined arm is then
matched. On another GPU the notebook also trains a space-joined control. About 25–40 minutes per arm.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = "https://github.com/sttruji/dynamic-state-anchoring.git"
REPO_REF = "main"  # A commit SHA reproduces an exact implementation.
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
GPU = subprocess.check_output([PYTHON, "-c", "import torch; print(torch.cuda.get_device_name(0))"], text=True).strip()
print("GPU:", GPU)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = str(DRIVE / "cache/huggingface")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
RESULTS = DRIVE / "results"
for path in Path("results").rglob("*"):  # Seed Drive results with the committed rows for comparison.
    target = RESULTS / path.relative_to("results")
    if path.is_file() and not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
WORK = Path("/content/format-test")      # Fast local disk; final models are copied to Drive.
SAVED = DRIVE / "format-test"
WORK.mkdir(exist_ok=True); SAVED.mkdir(parents=True, exist_ok=True)

def run(*args):
    subprocess.run([PYTHON, *map(str, args)], check=True)

In [ ]:
run("scripts/setup_data.py", "--eval-only")  # Writes and verifies data/gsm_*.json too.
run("scripts/smoke_pretrained_dsa.py", "--device", "cuda", "--output", WORK / "smoke-cuda.json", "--max-new-tokens", 4)

In [ ]:
import hashlib
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
EXPECTED = {  # Local preparation on 25 Oct; space reproduces the original Colab bundle byte for byte.
    "space": "63a2b505a54bea52a248662799107e7ac66af50084505eac1bbf9360c016981d",
    "newline": "7932a465ce22d801e60b5ca939a335cc043963300944bba737118dd9e77bdb50",
}
SELECTIONS = "152c34fab8a8903e126f73e7fcf84e7822c7dfe479e8f3958b114c812131301a"  # Same rows, same order.

def prepare(join):
    out = WORK / f"prepared-{join}"
    run("scripts/prepare_dsa_pilot.py", "--source-dir", os.environ["DSA_DATA_DIR"], "--output-dir", out,
        "--train-limit", 512, "--loops", 8, "--seed", 7, "--steps-join", join)
    assert sha(out / "train.tokenized.jsonl") == EXPECTED[join], f"{join} training data differs"
    assert sha(out / "selections.json") == SELECTIONS
    return out

def train(join):
    out = WORK / "runs" / f"learned-{join}"
    run("scripts/train_dsa.py", "--prepared-dir", prepare(join), "--output-dir", out,
        "--smoke-report", WORK / "smoke-cuda.json", "--device", "cuda",
        "--recurrence-mode", "flt_memory", "--write-mode", "learned", "--loops", 8, "--workspace", 4,
        "--steps", 1024, "--seed", 7, "--learning-rate", 3e-5, "--gradient-accumulation", 1, "--save-every", 256)
    return out

def evaluate(name, checkpoint, *extra):
    run("scripts/evaluate.py", "--model", "dsa", "--arg", f"checkpoint={checkpoint}", "--name", name,
        *extra, "--suite", "dev,valid", "--results-dir", RESULTS)

In [ ]:
newline = train("newline")
ORIGINAL_INITIAL = "6e0be5d178c6b74ec1deddd6faa1d2805f5ea43008f403461862aacfc2fbd211"  # Original T4 arm.
initial = json.loads((newline / "run_manifest.json").read_text())["initial_state_sha256"]
MATCHED = initial == ORIGINAL_INITIAL
print("Initial state matches the original space-joined arm:", MATCHED)
evaluate("dsa-data512-newline-learned-k8", newline)
evaluate("dsa-data512-newline-learned-k8-explicit", newline, "--arg", "route=explicit")
shutil.copytree(newline / json.loads((newline / "latest.json").read_text())["path"] / "model",
                SAVED / "learned-newline-model", dirs_exist_ok=True)

In [ ]:
# Only when the initial state differs (not a T4, or a different CUDA RNG): train the space-joined
# control here so the two arms share device and initialization.
if not MATCHED:
    space = train("space")
    evaluate("dsa-data512-space-learned-k8-rerun", space)
    evaluate("dsa-data512-space-learned-k8-rerun-explicit", space, "--arg", "route=explicit")
else:
    print("Matched to the saved arm: compare with dsa-data512-learned-k8 and dsa-data512-learned-k8-explicit.")

Compare the newline rows with `dsa-data512-learned-k8` / `-explicit` (or the rerun rows) and with
`gpt2-sft`, the checkpoint before DSA training. To share, copy the new folders under Drive
`results/runs/` and `results/predictions/` into your clone's `results/` and commit.

In [ ]:
run("scripts/evaluate.py", "--table", "--results-dir", RESULTS)